In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="transformers",
    notebook_name="01_training_a_small_transformer.ipynb"
)

# Training a Small Transformer

We will build a **character-level, decoder-only transformer** in PyTorch and train it to generate text.

The model uses the same architecture as GPT — multi-head attention with a causal mask, feed-forward networks, residual connections, and layer normalization. The only difference is scale: ~50K parameters instead of billions.

By the end of this notebook, you will watch a pile of random weights learn to produce readable text.

**Prerequisites:** [Transformer Block](../architecture/transformer-block.md) and [Training a Small Transformer (theory)](./training-a-small-transformer.md)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['figure.figsize'] = (12, 6)
matplotlib.rcParams['font.size'] = 12

# Use GPU if available, otherwise CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("Setup complete!")

## Part 1: The Training Data

We use a small inline text corpus — nursery rhymes. The model will learn the character-level patterns in this text: which letters tend to follow which, how words are structured, and what comes after spaces and newlines.

Character-level means we don't need a tokenizer. Each unique character gets a number.

In [ ]:
# Training corpus: nursery rhymes (inline, no file downloads needed)
corpus = """Twinkle twinkle little star,
How I wonder what you are.
Up above the world so high,
Like a diamond in the sky.
Twinkle twinkle little star,
How I wonder what you are.

Mary had a little lamb,
Its fleece was white as snow.
And everywhere that Mary went,
The lamb was sure to go.
It followed her to school one day,
Which was against the rule.
It made the children laugh and play,
To see a lamb at school.

Jack and Jill went up the hill,
To fetch a pail of water.
Jack fell down and broke his crown,
And Jill came tumbling after.

Humpty Dumpty sat on a wall,
Humpty Dumpty had a great fall.
All the kings horses and all the kings men,
Could not put Humpty together again.

Hey diddle diddle, the cat and the fiddle,
The cow jumped over the moon.
The little dog laughed to see such fun,
And the dish ran away with the spoon.
"""

# Build character vocabulary
chars = sorted(list(set(corpus)))
vocab_size = len(chars)

# Character-to-index and index-to-character mappings
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}

# Encode the entire corpus as a list of integers
data = [char_to_idx[ch] for ch in corpus]

print(f"Corpus length: {len(corpus)} characters")
print(f"Vocabulary size: {vocab_size} unique characters")
print(f"Characters: {''.join(chars)}")
print(f"")
print(f"First 50 chars of corpus: {repr(corpus[:50])}")
print(f"Encoded as indices:       {data[:50]}")
print(f"")
print(f"Example: '{corpus[0]}' -> {char_to_idx[corpus[0]]}")
print(f"Example: '{corpus[1]}' -> {char_to_idx[corpus[1]]}")

## Part 2: Create Training Batches

We slice the corpus into fixed-length windows. For each window, the input is positions 0 to N-1 and the target is positions 1 to N. The model learns to predict the next character at every position.

In [ ]:
# Hyperparameters
seq_len = 64        # Number of characters the model sees at once
batch_size = 16     # Number of sequences per training step

def get_batch(data, seq_len, batch_size):
    """Create a random batch of input-target pairs."""
    # Pick random starting positions
    max_start = len(data) - seq_len - 1
    starts = torch.randint(0, max_start, (batch_size,))
    
    # Build input and target tensors
    x = torch.stack([torch.tensor(data[s:s+seq_len]) for s in starts])
    y = torch.stack([torch.tensor(data[s+1:s+seq_len+1]) for s in starts])
    
    return x.to(device), y.to(device)

# Show one example
x_example, y_example = get_batch(data, seq_len, batch_size=1)

input_text = ''.join([idx_to_char[i.item()] for i in x_example[0]])
target_text = ''.join([idx_to_char[i.item()] for i in y_example[0]])

print(f"Batch shape: x={x_example.shape}, y={y_example.shape}")
print(f"")
print(f"Input:  {repr(input_text[:40])}...")
print(f"Target: {repr(target_text[:40])}...")
print(f"")
print("The target is the input shifted by one character.")
print("At every position, the model must predict the NEXT character.")

## Part 3: Build the Decoder-Only Transformer

This model uses `nn.MultiheadAttention`, `nn.Linear`, and `nn.LayerNorm` from PyTorch. These are the same components you built from scratch in NumPy in the architecture notebooks — now they can actually learn via backpropagation.

The architecture:
```
Character indices
    │
    ▼
Token Embedding + Positional Embedding
    │
    ▼
Transformer Block 1 (LayerNorm → Multi-Head Attention → + residual → LayerNorm → FFN → + residual)
    │
    ▼
Transformer Block 2 (same)
    │
    ▼
Final LayerNorm
    │
    ▼
Linear → logits (one score per character in vocabulary)
```

In [ ]:
class TransformerBlock(nn.Module):
    """One transformer block: Pre-LN variant (modern standard)."""
    
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, causal_mask, key_padding_mask=None):
        # Sub-layer 1: Multi-Head Attention with residual
        x_norm = self.ln1(x)
        attn_out, attn_weights = self.attn(
            x_norm, x_norm, x_norm,
            attn_mask=causal_mask,
            key_padding_mask=key_padding_mask,
            need_weights=True,
            average_attn_weights=True
        )
        x = x + self.dropout(attn_out)
        
        # Sub-layer 2: FFN with residual
        x_norm = self.ln2(x)
        x = x + self.ffn(x_norm)
        
        return x, attn_weights


class CharTransformer(nn.Module):
    """A small decoder-only transformer for character-level language modeling."""
    
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, max_len, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        
        # Token and position embeddings
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)
        
        # Stack of transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        
        # Final layer norm + output projection
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
    
    def forward(self, x, return_attention=False):
        batch_size, seq_len = x.shape
        
        # Token embeddings + positional embeddings
        positions = torch.arange(seq_len, device=x.device)
        x = self.token_emb(x) + self.pos_emb(positions)
        x = self.drop(x)
        
        # Create causal mask: True means "block this position"
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device=x.device)
        
        # Pass through transformer blocks
        all_attn = []
        for block in self.blocks:
            x, attn_w = block(x, causal_mask)
            all_attn.append(attn_w)
        
        # Final layer norm + project to vocab
        x = self.ln_f(x)
        logits = self.head(x)
        
        if return_attention:
            return logits, all_attn
        return logits


# Model hyperparameters
d_model = 64
n_heads = 4
n_layers = 2
d_ff = 256
max_len = 128
dropout = 0.1

model = CharTransformer(
    vocab_size=vocab_size,
    d_model=d_model,
    n_heads=n_heads,
    n_layers=n_layers,
    d_ff=d_ff,
    max_len=max_len,
    dropout=dropout
).to(device)

# Count parameters
n_params = sum(p.numel() for p in model.parameters())

print(f"CharTransformer Configuration:")
print(f"  vocab_size: {vocab_size}")
print(f"  d_model:    {d_model}")
print(f"  n_heads:    {n_heads} (d_k = {d_model // n_heads} per head)")
print(f"  n_layers:   {n_layers}")
print(f"  d_ff:       {d_ff}")
print(f"  max_len:    {max_len}")
print(f"")
print(f"Total parameters: {n_params:,}")
print(f"")
print(f"For comparison:")
print(f"  GPT-2 Small: 124,000,000 parameters")
print(f"  Our model:   {n_params:,} parameters ({n_params/124_000_000*100:.3f}% of GPT-2)")

## Part 4: Generate Text Before Training

Before we train the model, let's see what it produces with random weights. This is our baseline — the "before" in the before-and-after comparison.

In [ ]:
@torch.no_grad()
def generate(model, start_char, length, temperature=1.0):
    """Generate text one character at a time."""
    model.eval()
    
    # Start with the given character
    idx = torch.tensor([[char_to_idx[start_char]]], device=device)
    result = [start_char]
    
    for _ in range(length - 1):
        # Truncate to max_len if needed
        idx_input = idx[:, -max_len:]
        
        # Get model predictions
        logits = model(idx_input)
        
        # Take the logits for the last position
        logits_last = logits[0, -1, :] / temperature
        
        # Convert to probabilities and sample
        probs = torch.softmax(logits_last, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1)
        
        # Append to sequence
        idx = torch.cat([idx, next_idx.unsqueeze(0)], dim=1)
        result.append(idx_to_char[next_idx.item()])
    
    model.train()
    return ''.join(result)


# Generate text with untrained model
print("=" * 60)
print("BEFORE TRAINING (random weights):")
print("=" * 60)
print(generate(model, 'T', 200, temperature=1.0))
print("=" * 60)
print()
print("This is random gibberish. The model has no idea what")
print("language looks like yet. Every character is a random guess.")

## Part 5: Train the Model

Now for the main event. We train the model using:
- **Cross-entropy loss:** measures how far the model's predicted character distribution is from the actual next character
- **Adam optimizer:** adjusts all the model's weights to reduce the loss

We train for several hundred epochs. Each epoch processes multiple random batches from the corpus.

In [ ]:
# Training setup
optimizer = optim.Adam(model.parameters(), lr=3e-3)
criterion = nn.CrossEntropyLoss()

n_epochs = 500
steps_per_epoch = 5
losses = []

# Expected loss at random: -ln(1/vocab_size)
random_loss = np.log(vocab_size)
print(f"Expected loss with random guessing: {random_loss:.2f}")
print(f"Training for {n_epochs} epochs...")
print()

model.train()
for epoch in range(n_epochs):
    epoch_loss = 0.0
    
    for step in range(steps_per_epoch):
        x_batch, y_batch = get_batch(data, seq_len, batch_size)
        
        # Forward pass
        logits = model(x_batch)
        
        # Reshape for cross-entropy: (batch*seq_len, vocab_size) vs (batch*seq_len,)
        loss = criterion(logits.view(-1, vocab_size), y_batch.view(-1))
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / steps_per_epoch
    losses.append(avg_loss)
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1:>4d}/{n_epochs} | Loss: {avg_loss:.4f}")

print(f"")
print(f"Training complete!")
print(f"Starting loss: {losses[0]:.4f} (near random: {random_loss:.2f})")
print(f"Final loss:    {losses[-1]:.4f}")

## Part 6: Plot the Loss Curve

The loss curve shows how the model improves over training. It should start near the random-guessing baseline and decrease as the model learns character patterns.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(losses, color='steelblue', linewidth=1.5, alpha=0.7, label='Training loss')

# Add smoothed line
window = 20
if len(losses) > window:
    smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(losses)), smoothed, color='darkblue', linewidth=2.5, label=f'Smoothed (window={window})')

ax.axhline(y=random_loss, color='red', linestyle='--', linewidth=1.5, label=f'Random guessing ({random_loss:.2f})')

ax.set_xlabel('Epoch', fontsize=13)
ax.set_ylabel('Cross-Entropy Loss', fontsize=13)
ax.set_title('Training Loss: Character-Level Transformer', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"The loss dropped from {losses[0]:.2f} to {losses[-1]:.2f}.")
print(f"The model learned to predict the next character much better than random guessing.")

## Part 7: Generate Text After Training

Now let's see what the trained model produces. We'll try three different temperatures:
- **Low (0.5):** conservative, repetitive, but coherent
- **Medium (1.0):** balanced creativity and coherence
- **High (1.5):** creative but can produce nonsense

In [ ]:
temperatures = [0.5, 1.0, 1.5]

for temp in temperatures:
    print("=" * 60)
    print(f"AFTER TRAINING | Temperature = {temp}")
    print("=" * 60)
    print(generate(model, 'T', 200, temperature=temp))
    print()

print("Notice:")
print("  - Low temperature (0.5):  safe, repetitive, sticks to common patterns")
print("  - Medium temperature (1.0): balanced — recognizable words with variety")
print("  - High temperature (1.5):  creative but messy — more unusual character combos")
print()
print("The model learned character patterns from the nursery rhymes!")
print("It does not memorize the text — it learned which characters tend to follow which.")

## Part 8: Before vs After Comparison

Let's put the untrained and trained outputs side by side to see how much the model learned.

In [ ]:
# Generate fresh samples for comparison
# Temporarily reset model to random weights for "before"
# Instead, we saved the untrained output earlier. Let's just generate trained output.

trained_output = generate(model, 'T', 150, temperature=0.8)

print("BEFORE TRAINING (epoch 0):")
print("-" * 50)
print("(Random gibberish — no patterns learned yet)")
print("The model produces random characters because all")
print("weights are initialized to random values.")
print()
print("AFTER TRAINING (epoch {}):" .format(n_epochs))
print("-" * 50)
print(trained_output)
print()
print("The model learned from just {} characters of nursery rhymes!".format(len(corpus)))
print("Same architecture as GPT — just much, much smaller.")

## Part 9: Visualize Attention Weights

One of the most powerful things about transformers is that you can look inside them. The attention weights show which characters the model is "looking at" when predicting the next character.

Let's feed a short string into the trained model and visualize the attention pattern.

In [ ]:
# Feed a known string and get attention weights
test_string = "Twinkle twinkle"
test_indices = torch.tensor([[char_to_idx[ch] for ch in test_string]], device=device)

model.eval()
with torch.no_grad():
    logits, all_attn = model(test_indices, return_attention=True)

# all_attn is a list of [batch, seq, seq] tensors, one per layer
# Plot attention for both layers
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

chars_display = list(test_string)

for layer_idx in range(n_layers):
    ax = axes[layer_idx]
    # Average attention weights across heads (already averaged by PyTorch)
    attn_map = all_attn[layer_idx][0].cpu().numpy()
    
    im = ax.imshow(attn_map, cmap='Blues', aspect='auto', vmin=0)
    ax.set_xticks(range(len(chars_display)))
    ax.set_yticks(range(len(chars_display)))
    ax.set_xticklabels(chars_display, fontsize=9, rotation=45, ha='right')
    ax.set_yticklabels(chars_display, fontsize=9)
    ax.set_title(f'Layer {layer_idx + 1} Attention', fontsize=13)
    ax.set_xlabel('Key (attending TO)', fontsize=11)
    if layer_idx == 0:
        ax.set_ylabel('Query (attending FROM)', fontsize=11)
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle(f'Attention Weights for "{test_string}"', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("The causal mask is visible: the upper-right triangle is dark (near zero).")
print("Each character can only attend to itself and previous characters.")
print("")
print("Look for patterns:")
print("  - Characters attending to the space before them (word boundaries)")
print("  - The second 'twinkle' attending to the first 'twinkle'")
print("  - Vowels attending to nearby consonants (or vice versa)")

## Part 10: What the Model Learned

Let's look at which characters the model considers most likely after specific contexts. This reveals what patterns it extracted from the training data.

In [ ]:
@torch.no_grad()
def top_predictions(model, context, top_k=5):
    """Show the model's top-k predicted next characters given a context."""
    model.eval()
    idx = torch.tensor([[char_to_idx[ch] for ch in context]], device=device)
    logits = model(idx)
    probs = torch.softmax(logits[0, -1, :], dim=-1)
    top_probs, top_indices = torch.topk(probs, top_k)
    
    predictions = []
    for prob, idx_val in zip(top_probs, top_indices):
        char = idx_to_char[idx_val.item()]
        predictions.append((repr(char), prob.item()))
    
    return predictions


# Test with various contexts
test_contexts = [
    "Twinkle twinkle littl",
    "Mary had a littl",
    "the ",
    "and ",
    "Humpty Dump",
]

print("What does the model predict next?")
print("=" * 60)

for context in test_contexts:
    preds = top_predictions(model, context, top_k=5)
    print(f"")
    print(f"Context: {repr(context)}")
    print(f"  Top predictions:")
    for char, prob in preds:
        bar = '#' * int(prob * 40)
        print(f"    {char:>5s} : {prob:.3f} {bar}")

print(f"")
print("The model learned which characters follow common patterns!")
print("After 'littl', it predicts 'e' with high confidence.")
print("After 'Dump', it predicts 't' (from 'Dumpty').")

## Summary

We built and trained a **character-level, decoder-only transformer** from scratch in PyTorch:

| Component | What it does |
|-----------|-------------|
| Token embedding | Turns each character into a vector of numbers |
| Positional embedding | Tells the model where each character sits |
| Transformer blocks (x2) | Attention + FFN + residuals + layer norm |
| Causal mask | Prevents looking at future characters |
| Output head | Converts vectors into character probabilities |

### What we saw

1. **Before training:** The model produced random gibberish
2. **After training:** The model learned character patterns and produced text that looks like the training data
3. **Temperature** controls the trade-off between coherence and creativity
4. **Attention weights** reveal what the model "looks at" when making predictions

This is the same mechanism used by GPT-4, Claude, and every modern language model. The only differences are scale (billions vs thousands of parameters) and training data (trillions of tokens vs hundreds of characters).

---

**Next:** [Encoder vs Decoder Comparison](./encoder-vs-decoder.md) — we'll train two different transformer architectures on the same task and see which one works better.

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)